<a href="https://colab.research.google.com/github/winnerthetechgenie/KNN--Diabetes-Prediction/blob/main/KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KNN - Predict whether a person will be diagnosed with diabetes or not

In [192]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import io

In [193]:
#Here we import the dataset into colab
from google.colab import files
uploaded = files.upload()

Saving diabetes.csv to diabetes (1).csv


In [194]:
data = pd.read_csv(io.BytesIO(uploaded['diabetes.csv']))

In [195]:
print(data.shape)
print(data.columns)
data.head()

(768, 9)
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Checking for null values

In [196]:
data.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

The data doesnt contain null values

so we check for zero values

In [197]:
data['Outcome'].value_counts()

0    500
1    268
Name: Outcome, dtype: int64

In [198]:
cols = data.columns
for col in cols:
    print(col + ':', data[data[col]==0].shape[0])

Pregnancies: 111
Glucose: 5
BloodPressure: 35
SkinThickness: 227
Insulin: 374
BMI: 11
DiabetesPedigreeFunction: 0
Age: 0
Outcome: 500


the output above shows the columns and their respective number of zero values

Considering the fact we are trying to predict diabetes outcome, the glucose, bloodpressure, skinthickness, insulin and BMI cannot have a value of 0.

it only implies the values were not inputted so we fill it with the mean of the respective columns

In [199]:
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_columns:
  data[col] = data[col].replace(0, np.NaN)
  mean = data[col].mean(skipna=True)
  data[col] = data[col].replace(np.NaN, mean)

checking if the columns has been repaced with the mean

In [200]:
for col in zero_columns:
  print(col, data[data[col]==0].shape[0])

Glucose 0
BloodPressure 0
SkinThickness 0
Insulin 0
BMI 0



# VISUALIZATION



In [201]:
import matplotlib.pyplot as plt
import seaborn as sns

Scalling the dataset

### Scaling and training

In [202]:
#Splitting the dataset into the dependent and independent features

x = data.iloc[:, :-1]
y = data.iloc[:, -1]

x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=0, test_size=0.2)
y_train.value_counts()

0    393
1    221
Name: Outcome, dtype: int64

In [203]:
#Here, the dependent variables are scaled to prevent bias among the features
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)
'''x_test doesnt use 'fit_transform' rather 'transform' so that the same format used to scale 
x_train is used for the x_test'''

"x_test doesnt use 'fit_transform' rather 'transform' so that the same format used to scale \nx_train is used for the x_test"

# modelling

In [204]:
#to know the best value for k, we perform a squareroot

import math
print(math.sqrt(len(x_test)))
output: 28
#it is best an odd number is used, so 1 is removed


12.409673645990857


In [205]:
model = KNeighborsClassifier(n_neighbors=11, p=2, metric='euclidean')

model.fit(x_train, y_train)

KNeighborsClassifier(metric='euclidean', n_neighbors=11)

In [206]:
y_pred = model.predict(x_test)

y_pred

array([1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [207]:
confusion_matrix(y_test, y_pred)

array([[94, 13],
       [16, 31]])

In [213]:
print((accuracy_score(y_test, y_pred)*100).round(2))
print((f1_score(y_test,y_pred)*100).round(2))

81.17
68.13
